In [ ]:
#4. Heatmap of prisoner deaths relative to size

def prison_pop_heat(df):
    chart = alt.Chart(df, title="Inmates Deaths Relative to Inmate and Staff Counts")
    prison_pop_heat = chart.mark_rect().transform_aggregate(
        count="count()",
        staff_confirmed="average(Staff.Confirmed)",
        residents_deaths="average(Residents.Deaths)",
        groupby=["Current Facility"]
    ).encode(
        alt.X("count:Q", title="Prisoner Count"),
        alt.Y("staff_confirmed:Q", title="Staff Confirmed"),
        alt.Color("residents_deaths:Q", title="Inmate Deaths", scale=alt.Scale(scheme="viridis"))
    )

    return prison_pop_heat

prison_pop_heat(combined_df)

In [ ]:
def race_sentence(df):
    chart = alt.Chart(df, title="Comparison of Sentence Length Among Black, White, and Hispanic Inmates")
    domain = ["B", "H", "W"]
    range = ["#44414f","#686576", "#b0a1a1"]
    race_sentence_chart = chart.mark_area().encode(
        alt.X("Age:Q"),
        alt.Y("average(Sentence (Years)):Q"),
        alt.Color("Race:N").scale(domain=domain, range=range)
    ).transform_filter(
        (alt.datum["Sentence (Years)"] < 100),
        (alt.datum["Age"] < 100)
    )

    return race_sentence_chart

#find way to change to full races?

race_sentence(race_comparison_df)

In [ ]:
breaks = [16, 20, 25, 30, 40, 50, 60, 70, 80]
labels = [
    "<16", "16-20", "20-25", "25-30", "30-40",
    "40-50", "50-60", "60-70", "70-80", "80+"
]

binned_female_prisoners = prisoner_df.with_columns(
    pl.col("Age").cut(breaks, labels=labels).alias("Age_bucket")
)

binned_female_prisoners = binned_female_prisoners.filter(
    (pl.col("Gender") == "F")
)

#8. radial plot for sentence time for women
binned_female_prisoners = binned_female_prisoners.with_columns(
    pl.col("Age_bucket").cast(pl.String)
)

def women_sentence_times(df):
    df = df.filter(
        pl.col('Sentence (Years)').cast(pl.Float64, strict=False).is_not_null()
    )
    chart = alt.Chart(df, title="Comparison of Sentence Lengths Among Female Inmates by Age Groups")
    base = chart.transform_aggregate(
            avg_sentence="average(Sentence (Years))",
            groupby=["Age_bucket"]
            ).encode(
            alt.Theta("avg_sentence:Q").stack(True),
            alt.Radius("avg_sentence:Q").scale(type="sqrt", zero=True, rangeMin=20),
            alt.Color("Age_bucket:N", legend=alt.Legend(title="Age Group"), scale=alt.Scale(scheme="viridis")),
            )

    c1 = base.mark_arc(innerRadius=20, stroke="#fff")

    c2 = base.mark_text(radiusOffset=10).encode(
        text=alt.Text("avg_sentence:Q", format='.1f'),
        color=alt.value("black"))

    final_radial = c1 + c2
    
    return final_radial

women_sentence_times(binned_female_prisoners)   